# Chapter 03: Forty pages

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "03"
__file__ = os.path.abspath("run.py")

## The scene

Digital forensics delivers Reyes's personal notes: thirty-seven entries, January to the night he
died, most of them about fuel prices and payroll. Varma drops the printout on your desk.
"Somewhere in there he says what he found. Feed the lot to LANTERN and ask." You paste the whole
file into one prompt and ask. LANTERN answers at once, and with confidence: Reyes never mentions
Northwind. You have read the anonymous envelope. You know that cannot be true.

## The concept

A model reads tokens, not pages, and it can only attend to a fixed number of them at once. That
number is the **context window**. Everything the model knows about your request (system prompt,
documents, conversation history, the question) and everything it writes in reply must fit inside
it together.

```mermaid
flowchart LR
    subgraph W["Context window: num_ctx tokens"]
      direction LR
      S[system] --> D[documents] --> H[history] --> Q[question] --> A[answer being written]
    end
    X["Anything that does not fit"] -.->|"dropped before the model sees it"| W
```

There are two limits, and they are easy to confuse:

| Limit | Value for qwen3:1.7b | Who sets it |
|---|---|---|
| Trained context length | 40,960 tokens | The model's training. The most it can handle at all. |
| `num_ctx` | 8,192 in this lab | The server, per request. The window actually allocated. |

Servers allocate less than the maximum because memory for the window (the KV cache) grows with
`num_ctx` for every request. If you send no `num_ctx`, Ollama uses its own default, which is
smaller still.

## How it works

Reyes's notes are 4,942 words: 6,370 tokens with the question. The line that matters, "Northwind
is not a vendor", sits 50% of the way through, in the entry for 6 February.

![What happens to the notes in each window](../../docs/diagrams/03-context-window.svg)

The script runs four experiments on the same file and the same question:

| Experiment | Tokens the model saw | What happened |
|---|---|---|
| `num_ctx = 2048` | 1,026 | Ollama silently kept only the end. The 6 February entry was gone. |
| `num_ctx = 8192` | 6,370 | Everything fitted. The model still answered NOT FOUND. |
| First quarter + last quarter, `num_ctx = 8192` | 3,202 | The application cut the middle out. NOT FOUND, correctly, for what it was given. |
| Four chunks, one question each | ~1,300 to 1,800 each | Chunk 2 returned the exact quote. |

```mermaid
flowchart TB
    N["reyes-notes.md<br/>6,370 tokens"] --> C1["chunk 1<br/>entries 1 to 9"]
    N --> C2["chunk 2<br/>entries 10 to 19"]
    N --> C3["chunk 3<br/>entries 20 to 29"]
    N --> C4["chunk 4<br/>entries 30 to 37"]
    C1 --> R1[NOT FOUND]
    C2 --> R2["'Northwind is not a vendor...'"]
    C3 --> R3[NOT FOUND]
    C4 --> R4[NOT FOUND]
    R1 & R2 & R3 & R4 --> M[Combine: one hit, with its quote]
```

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import re
import time

from lantern import case, chat, learn, load_config
from lantern.config import tier
from lantern.llm import context_length
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    reply,
    save_shot,
    step,
    your_turn,
)

chapter("03", "Forty pages", """
Digital forensics delivers Reyes's personal notes: thirty-seven entries, January to the night
he died, most of them about fuel prices and payroll. Varma drops the printout on your desk.
"Somewhere in there he says what he found. Feed the lot to LANTERN and ask." You paste the
whole file into one prompt and ask. LANTERN answers at once, and with confidence: Reyes never
mentions Northwind. You have read the anonymous envelope. You know that cannot be true.
""")

notes = case.read("reyes-notes.md")
entries = notes.count("\n## ")
pos = notes.index("Northwind is not a vendor")
evidence("reyes-notes.md", f"{entries} dated entries, {len(notes.split()):,} words. "
         f"The sentence 'Northwind is not a vendor' starts at character {pos:,} of "
         f"{len(notes):,}: {pos / len(notes):.0%} of the way through.")
QUESTION = ("\n\nQuestion: What does Reyes say about Northwind Freight? Quote his exact words. "
            "If the notes do not mention Northwind, say NOT FOUND.")
note("The question is the `QUESTION` variable in run.py. Every step sends it after the notes.")

### 1. Two different limits

In [ ]:
step("1. Two different limits")
cfg = load_config()["generation"]
trained = context_length()
t = Table()
t.add_column("limit")
t.add_column("tokens", justify="right")
t.add_column("set by")
t.add_row("trained context length", f"{trained:,}", "the model's training (read from its metadata)")
t.add_row("num_ctx (this lab's default)", f"{cfg['num_ctx']:,}",
          "config.yaml, sent with every call")
t.add_row("Ollama's own default", "4,096 or less", "used when you send no num_ctx at all")
console.print(t)
note("The window holds the prompt AND the answer being written. Memory grows with num_ctx, "
     "which is why servers keep it smaller than the model could handle.")



def found(text: str) -> bool:
    """Did a reply give Reyes's line? (It must say 'not a vendor', not just name Northwind.)"""
    return "not a vendor" in text.lower()

### 2. The whole file in a 2,048 token window

In [ ]:
step("2. The whole file in a 2,048 token window")
if tier() == "hosted":
    note("num_ctx is an Ollama setting. The hosted provider has one fixed window (1,000,000 "
         "tokens for this model) and rejects a prompt that does not fit rather than cutting it, "
         "so on this tier steps 2 and 3 send the same thing and nothing is dropped.")
p_seen = learn.predict_number(
    "The notes and the question are about 6,400 tokens. The window is 2,048. How many prompt "
    "tokens will the model actually see?",
    unit="tokens",
)
small = chat(notes + QUESTION, num_ctx=2048, temperature=0, max_tokens=200)
reply("num_ctx = 2048", small.text, meta=f"prompt tokens the model actually saw: "
      f"{small.prompt_tokens:,}", style="red")
note(f"Ollama kept {small.prompt_tokens:,} tokens and threw the rest away. No error, no "
     "warning in the reply. It drops from the FRONT, so the question at the end survived "
     "and the evidence did not.")
learn.reveal(
    p_seen,
    actual=small.prompt_tokens,
    tolerance=0.3,
    why="The window holds the prompt and the answer together, and Ollama keeps the most recent "
    "tokens, so the prompt is cut to about half the window. No error is raised.",
)

### 3. The whole file in an 8,192 token window

In [ ]:
step("3. The whole file in an 8,192 token window")
p_big = learn.predict(
    "Now every token fits. Will LANTERN find Reyes's line about Northwind?",
    ["Yes, it gives the line", "No, it misses it"],
)
t0 = time.perf_counter()
big = chat(notes + QUESTION, num_ctx=8192, temperature=0, max_tokens=200)
big_seconds = time.perf_counter() - t0
reply("num_ctx = 8192", big.text, meta=f"prompt tokens: {big.prompt_tokens:,}", style="yellow")
note("Every token fits now. Check what the model did with them. A line in the middle of a long "
     "prompt gets the least attention: 'lost in the middle'. Fitting is not the same as reading.")
learn.reveal(
    p_big,
    answer=0 if found(big.text) else 1,
    why="Fitting is not reading. The line sits in the middle of a long prompt, where a small model "
    "attends least, and the result changes with the wording of the question.",
)

### 4. An application that 'fits' the file by keeping the start and the end

In [ ]:
step("4. An application that 'fits' the file by keeping the start and the end")
quarter = len(notes) // 4
head_tail = notes[:quarter] + "\n\n[... middle of the document removed to fit ...]\n\n" + \
    notes[-quarter:]
cut = chat(head_tail + QUESTION, num_ctx=8192, temperature=0, max_tokens=200)
reply("first quarter + last quarter", cut.text, meta=f"prompt tokens: {cut.prompt_tokens:,}",
      style="yellow")
note("Keeping the start and end is a common way to squeeze a long document. It is exactly the "
     "wrong cut when the evidence is in the middle.")

### 5. Read it in pieces: one question per chunk

In [ ]:
step("5. Read it in pieces: one question per chunk")
CHUNKS = 4  # how many pieces to cut the notes into
OVERLAP = 1  # entries each chunk repeats from the one before (used by --mine)


def chunk_notes(text: str, chunks: int) -> list[str]:
    """The notes cut on entry boundaries into `chunks` pieces of whole entries."""
    parts = text.split("\n## ")
    size = len(parts) // chunks + 1
    return ["\n## ".join(parts[i * size:(i + 1) * size]) for i in range(chunks)]


def read_notes(text: str, question: str, *, chat=chat) -> dict:
    """The map step: the question asked of every chunk, one call each."""
    replies = []
    for chunk in chunk_notes(text, CHUNKS):
        r = chat(f"<notes>\n{chunk}\n</notes>{question}", num_ctx=8192, temperature=0,
                 max_tokens=150)
        replies.append({"chunk": chunk, "text": r.text, "prompt_tokens": r.prompt_tokens,
                        "found": "NOT FOUND" not in r.text})
    return {"replies": replies}


MINE = learn.mine()
p_chunks = learn.predict(
    f"The notes go in {CHUNKS} pieces, the same question to each. How many pieces will come back "
    "with Reyes's line?",
    ["None", "Exactly one", "More than one"],
)
t0 = time.perf_counter()
if MINE:
    size = len(notes.split("\n## ")) // CHUNKS + 1
    out = MINE.read_notes(notes, QUESTION, chat=chat, size=size, overlap=OVERLAP)
else:
    out = read_notes(notes, QUESTION)
chunk_seconds = time.perf_counter() - t0
n = len(out["replies"])
for i, r in enumerate(out["replies"]):
    reply(f"chunk {i + 1} of {n}",
          r["text"], meta=f"{r['prompt_tokens']:,} tokens", style="green" if r["found"] else "dim")
note(f"{CHUNKS} short prompts, each small enough to be read properly. This is the idea behind "
     "map-reduce summarising and, in a later case file, retrieval.")
hits = [r for r in out["replies"] if found(r["text"])]
flat_notes = " ".join(notes.split())
quoted = [q for r in hits for q in re.findall(r'["\u201c]([^"\u201d]{12,})["\u201d]', r["text"])]
verbatim = any("not a vendor" in q.lower() and " ".join(q.split()) in flat_notes for q in quoted)
learn.reveal(
    p_chunks,
    answer=min(len(hits), 2),
    why="Each chunk is short enough to be read properly, and the line sits in exactly one of them. "
    "A piece without it should say NOT FOUND; one that claims a hit without the line is a false "
    "alarm, which is why the reduce step checks the quote.",
)
learn.record(
    trained_context=trained,
    small_prompt_tokens=small.prompt_tokens,
    small_found=found(small.text),
    big_prompt_tokens=big.prompt_tokens,
    big_found=found(big.text),
    big_seconds=round(big_seconds, 1),
    headtail_prompt_tokens=cut.prompt_tokens,
    headtail_found=found(cut.text),
    chunks=n,
    chunk_hits=len(hits),
    chunk_found=bool(hits),
    chunk_quote_verbatim=verbatim,
    chunk_seconds=round(chunk_seconds, 1),
    mine=bool(MINE),
)

your_turn(
    "03",
    check=[
        "Compare the prompt tokens under each reply with the size of the notes. In which windows "
        "did the model see the whole file?",
        "Step 3 held every token. Did its answer find the line? Open case/reyes-notes.md, search "
        "for 'Northwind is not a vendor' and check any quote word for word.",
        "In step 4, which part of the notes was cut? Compare with where the evidence panel at "
        "the top says the Northwind sentence sits.",
    ],
    exercises=[
        "In run.py, change `QUESTION` to ask 'What does Reyes say about Northwind?' (drop "
        "'Freight'). Read step 3. Run it twice more. How stable is it?",
        "In run.py, in the `small = chat(...)` line, send QUESTION + notes in place of "
        "notes + QUESTION. What does the model answer when the question is what gets dropped?",
        "In run.py, in the `big = chat(...)` line, change num_ctx=8192 to num_ctx=6400 "
        "(max_tokens is already 200). How long is the answer?",
        "In run.py, step 5, change CHUNKS = 4 to CHUNKS = 8. Is it faster or slower overall? "
        "Does it still find the line?",
    ],
)

finding("""
In a 2,048 token window LANTERN saw a sixth of the notes and said Northwind was never
mentioned. In an 8,192 token window the whole file fitted, and it still missed the line in the
middle. Cutting the middle out made sure of it. Read in four pieces, the entry from 6 February
surfaced: "Northwind is not a vendor. It has no ships, no staff and no office, only a PO box in
Port Sabran." Every payment, Reyes wrote, was coded to Raj Malhotra's cost centre and dated
while Malhotra was in Port Sabran.

Varma reads it twice. "He knew five weeks before he died. Who else did?"
""")
save_shot("03-context-window")

## The nuances

**1. Overflow is silent.** With `num_ctx = 2048`, Ollama did not raise an error. It kept 1,026
tokens, threw away about 5,300, and the model answered with total confidence. The only trace is
`prompt_eval_count` in the response (`Reply.prompt_tokens` in this lab). Always compare it with
what you sent.

**2. Ollama drops from the front.** It keeps the most recent tokens, because in a chat the latest
turns usually matter most. For a document followed by a question, that means the question survives
and the start of the document does not. A model that can still see the question will happily
answer it from whatever is left.

**3. The window includes the answer.** `num_ctx` covers input and output together. A prompt of
8,000 tokens in an 8,192 window leaves 192 tokens to answer in. Budget for the reply (this lab
reserves `max_tokens: 800`), and remember that thinking tokens (chapter 10) come out of the same
budget.

**4. Fitting is not reading.** At 8,192 tokens the whole file fitted and the model still missed
the line. Models attend most reliably to the start and the end of a long context and least to the
middle. This is known as "lost in the middle". Small models suffer from it badly; large models
less, but not zero. The exact behaviour varies with wording: in an earlier run with a slightly
different question, the same model did find the line in the full file. That variability is the
problem.

**5. Truncating in your own code needs a strategy.** Keeping the first and last parts of a
document is a common shortcut. It is exactly wrong when the evidence is in the middle, and you
will not know where the evidence is in advance.

**6. Chunking trades one hard question for several easy ones.** Each chunk is short enough to be
read properly, and the answers are combined afterwards (this is map-reduce). The price: more calls,
and a fact that spans two chunks can be missed. Retrieval, which picks only the relevant chunks,
is the next step and gets its own case file.

**7. More window costs more.** Prompt processing time grows with the number of tokens. In this
run, on an Apple M1 Pro, the full 6,370 token prompt took 8.2 seconds, and all four chunks
together took 7.8. Hosted APIs charge per input token, so a bigger window is also a bigger bill.

**8. Tokens are not words.** 4,942 words became about 6,300 tokens here (roughly 1.3 tokens per
word for this English text). Numbers, code and other languages tokenise differently. Measure; do
not estimate.

## On a hosted model

```bash
make lesson N=03 TIER=hosted     # needs ANTHROPIC_API_KEY in .env and `uv sync --extra cloud`
make lesson N=03 TIER=hosted REPLAY=1   # once a run has been recorded, no key needed
```

The hosted path has no `num_ctx`: `context_length()` returns the 1,000,000-token window documented
for claude-haiku-5-5 (`hosted.anthropic.context_window` in `config.yaml`), and the provider rejects
a prompt that does not fit rather than cutting it. Run.py says so before step 2: on this tier
steps 2 and 3 send the same thing and nothing is dropped, so the 2,048-token truncation cannot be
reproduced. The head-and-tail and chunking steps still run. Claude does not take `temperature`,
`top_k`, `top_p` or `seed`; `cloud.py` leaves out `temperature=0` and reports it.

| Measure | Local: qwen3:1.7b, Ollama | Hosted: claude-haiku-5-5 |
|---|---|---|
| Trained context length (tokens) | 40,960 | not run |
| Prompt tokens seen, `num_ctx = 2048` | 1,026 | not run |
| Prompt tokens seen, `num_ctx = 8192` | 6,370 | not run |
| Prompt tokens, head and tail | 3,202 | not run |

What the hosted run will test: whether "NOT FOUND" with the whole file in view is a small-model
effect of reading a long prompt. That cutting
the middle out removes the answer, and that a quote and the facts beside it must both be checked,
should hold on both.

## Try this

1. Change the question to "What does Reyes say about Northwind?" (drop "Freight", add nothing
   else) and rerun step 3. Does the full file now work? Run it twice more. How stable is it?
2. Move the question to the start of the prompt, before the notes, in the 2,048 window. What does
   the model answer now that the question is the part that gets dropped?
3. Set `num_ctx` to 6,400 (just above the prompt) with `max_tokens` at 200. How long is the answer?
4. Split into 8 chunks instead of 4. Is it faster or slower overall? Does it still find the line?

## Master it

After this chapter you can tell the trained context length from the window, spot a silent
overflow, explain why fitting is not reading, and read a long document in overlapping chunks
whose hits are checked.

**Predict, then see.** `make lesson N=03` stops three times to ask what will happen: how many
tokens survive the 2,048 window, whether the full file finds the line once it all fits, and how
many chunks come back with it. Each answer is checked against your own run, not a fixed key. Add
`ARGS=--no-predict` to watch without stopping.

**Build it.** Write the chunker and the map step in
[`practice/exercise.py`](practice/exercise.py):

```bash
make exercise N=03             # 7 offline tests, scripted model, no Ollama needed
make hint N=03                 # one hint at a time
make solution N=03             # the reference, once you have tried
make lesson N=03 ARGS=--mine   # then step 5 of the live lesson reads with your code
```

**Check yourself.** `make quiz N=03`: seven questions, each with the reason behind the answer.

**Break it.** `make challenge N=03` checks your latest run against three targets: your reader on
the real notes; one chunk instead of four, so the line is lost again; and a cut that sends under
half the notes but keeps the evidence.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 03

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 03

In [ ]:
%run ../../scripts/learn.py hint 03

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 03

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

In a 2,048 token window LANTERN saw a sixth of the notes and said Northwind was never mentioned. In
an 8,192 token window the whole file fitted, and it still missed the line in the middle. Cutting
the middle out made sure of it. Read in four pieces, the entry from 6 February surfaced:
"Northwind is not a vendor. It has no ships, no staff and no office, only a PO box in Port
Sabran." Every payment, Reyes wrote, was coded to Raj Malhotra's cost centre and dated while
Malhotra was in Port Sabran.

Varma reads it twice. "He knew five weeks before he died. Who else did?"

**Next:** [Chapter 04: The briefing](../04-what-goes-in-context/README.md)